# Separate & Sequential Learning — Training + Evaluation

Trains 3 single-task BERT models then chains them in a sequential pipeline
to quantify error propagation vs. joint learning.

**Models:** CLS-only, BIO-only, REL-only (each with `task_loss_weights` zeroing other heads).
**Pipeline:** CLS → BIO → REL, no cleanup between stages.

In [1]:
import sys, os, json, copy
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
for d in [SRC_DIR, os.path.join(SRC_DIR, 'jointlearning')]:
    if d not in sys.path:
        sys.path.insert(0, d)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer

from jointlearning.model import JointCausalModel
from jointlearning.utility import compute_class_weights, label_value_counts, set_seed, seed_worker
from jointlearning.dataset_collator import CausalDataset, CausalDatasetCollator
from jointlearning.trainer import train_model
from jointlearning.evaluate_joint_causal_model import evaluate_model, print_eval_report
from analysis.llm2doccano import convert_llm_output_to_doccano
from analysis.causal_eval import evaluate, display_results

from cmp_config import *

set_seed(SEED)
print(f'Device: {DEVICE}  |  Seed: {SEED}  |  Encoder: {MODEL_CONFIG["encoder_name"]}')

Device: cuda  |  Seed: 8642  |  Encoder: bert-base-uncased


---
### Training configuration

In [2]:
print('=== Task Loss Weights (2 heads zeroed per model) ===')
for task, w in TASK_WEIGHTS.items():
    print(f'  {task.upper()}-only: {w}')

print(f'\n=== Training Config ===')
for k, v in TRAINING_CONFIG.items():
    print(f'  {k}: {v}')

print(f'\nModel save paths:')
for task, p in MODEL_PATHS.items():
    print(f'  {task}: {p}')

=== Task Loss Weights (2 heads zeroed per model) ===
  CLS-only: {'cls': 1.0, 'bio': 0.0, 'rel': 0.0}
  BIO-only: {'cls': 0.0, 'bio': 1.0, 'rel': 0.0}
  REL-only: {'cls': 0.0, 'bio': 0.0, 'rel': 1.0}

=== Training Config ===
  batch_size: 16
  num_epochs: 20
  learning_rate: 1e-05
  weight_decay: 0.1
  gradient_clip_val: 1.0
  patience_epochs: 10

Model save paths:
  cls: /home/rnorouzini/JointLearning/reviewer_extra_analysis/comparison_learning/models/cls/separate_cls.pt
  bio: /home/rnorouzini/JointLearning/reviewer_extra_analysis/comparison_learning/models/bio/separate_bio.pt
  rel: /home/rnorouzini/JointLearning/reviewer_extra_analysis/comparison_learning/models/rel/separate_rel.pt


In [3]:
# Load datasets
train_df = pd.read_csv(TRAIN_DATA_PATH)
val_df = pd.read_csv(VAL_DATA_PATH)
print(f'Train: {len(train_df)}  Val: {len(val_df)}')

train_dataset = CausalDataset(train_df, tokenizer_name='bert-base-uncased', max_length=DATASET_CONFIG['max_length'])
val_dataset = CausalDataset(val_df, tokenizer_name='bert-base-uncased', max_length=DATASET_CONFIG['max_length'])

labels_flat = label_value_counts(train_dataset)
cls_w = compute_class_weights(labels_flat['cls_labels_flat'], num_classes=2, technique='ens', ignore_index=-100)
bio_w = compute_class_weights(labels_flat['bio_labels_flat'], num_classes=7, technique='ens', ignore_index=-100)
rel_w = compute_class_weights(labels_flat['rel_labels_flat'], num_classes=2, technique='ens', ignore_index=-100)

collator = CausalDatasetCollator(tokenizer=train_dataset.tokenizer)
_g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=TRAINING_CONFIG['batch_size'], collate_fn=collator,
                          shuffle=True, generator=_g, worker_init_fn=seed_worker)
val_loader = DataLoader(val_dataset, batch_size=TRAINING_CONFIG['batch_size'], collate_fn=collator,
                        shuffle=False, worker_init_fn=seed_worker)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

Train: 2109  Val: 453


cls_labels_value_counts:
 1    1062
0    1047
Name: count, dtype: int64


bio_labels_value_counts:
  6      53237
 3       8900
 1       7077
-100     4218
 2       1356
 0       1215
 5        485
 4         81
Name: count, dtype: int64


rel_labels_value_counts:
 0    2956
1    1506
Name: count, dtype: int64
Train batches: 132  Val batches: 29


---
### Training helper — `make_eval_fn` overrides `overall_avg_f1` to use only the active task

In [4]:
def make_eval_fn(active_task: str):
    """Return an eval function whose overall_avg_f1 tracks only *active_task*.
    Without this, the 2 untrained heads contribute ~0 F1 and drag the average
    down, making model selection (early stopping / ReduceLROnPlateau) unreliable.
    """
    def _eval_fn(model, dataloader, device, id2label_cls, id2label_bio, id2label_rel):
        results = evaluate_model(model, dataloader, device, id2label_cls, id2label_bio, id2label_rel)
        task_key = {'cls': 'task_cls', 'bio': 'task_bio', 'rel': 'task_relation'}[active_task]
        tm = results.get(task_key, {})
        f1 = tm.get('macro avg', {}).get('f1-score', 0.0) if isinstance(tm, dict) else 0.0
        results['overall_avg_f1'] = f1
        return results
    return _eval_fn

print('make_eval_fn defined — wraps evaluate_model to track only the active task')

make_eval_fn defined — wraps evaluate_model to track only the active task


In [5]:
# Check if models already trained — skip if so (saves hours)
models_exist = all(os.path.exists(MODEL_PATHS[t]) for t in ['cls','bio','rel'])
if models_exist:
    print('All models already trained. Loading histories...')
    cls_hist = json.load(open(os.path.join(os.path.dirname(MODEL_PATHS['cls']), 'training_history.json')))
    bio_hist = json.load(open(os.path.join(os.path.dirname(MODEL_PATHS['bio']), 'training_history.json')))
    rel_hist = json.load(open(os.path.join(os.path.dirname(MODEL_PATHS['rel']), 'training_history.json')))
    SKIP_TRAINING = True
else:
    print('Training models from scratch...')
    SKIP_TRAINING = False

All models already trained. Loading histories...


---
## Train CLS-only model
`task_loss_weights = {cls:1.0, bio:0.0, rel:0.0}`

In [6]:
if not SKIP_TRAINING:
    print(f'Task weights: {TASK_WEIGHTS["cls"]}')
    set_seed(SEED)
    cls_model = JointCausalModel(**MODEL_CONFIG)
    opt = optim.AdamW(cls_model.parameters(), lr=TRAINING_CONFIG['learning_rate'], weight_decay=TRAINING_CONFIG['weight_decay'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.1, patience=2)

    cls_model, cls_hist = train_model(
        model=cls_model, train_dataloader=train_loader, val_dataloader=val_loader,
        optimizer=opt, num_epochs=TRAINING_CONFIG['num_epochs'], device=DEVICE,
        id2label_cls=id2label_cls, id2label_bio=id2label_bio, id2label_rel=id2label_rel,
        model_save_path=MODEL_PATHS['cls'], scheduler=sched,
        cls_class_weights=cls_w, bio_class_weights=bio_w, rel_class_weights=rel_w,
        patience_epochs=TRAINING_CONFIG['patience_epochs'], seed=SEED,
        max_grad_norm=TRAINING_CONFIG['gradient_clip_val'],
        eval_fn_metrics=make_eval_fn('cls'), print_report_fn=print_eval_report,
        task_loss_weights=TASK_WEIGHTS['cls'],
    )
    del cls_model
    torch.cuda.empty_cache()
else:
    print('CLS model already trained — skipped')

print(f'CLS-only best val F1: {max(cls_hist["val_overall_f1"]):.4f}')

CLS model already trained — skipped
CLS-only best val F1: 0.8117


---
## Train BIO-only model
`task_loss_weights = {cls:0.0, bio:1.0, rel:0.0}`

In [7]:
if not SKIP_TRAINING:
    print(f'Task weights: {TASK_WEIGHTS["bio"]}')
    set_seed(SEED)
    bio_model = JointCausalModel(**MODEL_CONFIG)
    opt = optim.AdamW(bio_model.parameters(), lr=TRAINING_CONFIG['learning_rate'], weight_decay=TRAINING_CONFIG['weight_decay'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.1, patience=2)

    bio_model, bio_hist = train_model(
        model=bio_model, train_dataloader=train_loader, val_dataloader=val_loader,
        optimizer=opt, num_epochs=TRAINING_CONFIG['num_epochs'], device=DEVICE,
        id2label_cls=id2label_cls, id2label_bio=id2label_bio, id2label_rel=id2label_rel,
        model_save_path=MODEL_PATHS['bio'], scheduler=sched,
        cls_class_weights=cls_w, bio_class_weights=bio_w, rel_class_weights=rel_w,
        patience_epochs=TRAINING_CONFIG['patience_epochs'], seed=SEED,
        max_grad_norm=TRAINING_CONFIG['gradient_clip_val'],
        eval_fn_metrics=make_eval_fn('bio'), print_report_fn=print_eval_report,
        task_loss_weights=TASK_WEIGHTS['bio'],
    )
    del bio_model
    torch.cuda.empty_cache()
else:
    print('BIO model already trained — skipped')

print(f'BIO-only best val F1: {max(bio_hist["val_overall_f1"]):.4f}')

BIO model already trained — skipped
BIO-only best val F1: 0.4916


---
## Train REL-only model
`task_loss_weights = {cls:0.0, bio:0.0, rel:1.0}`

Uses gold spans from dataset for relation pair construction during training.

In [8]:
if not SKIP_TRAINING:
    print(f'Task weights: {TASK_WEIGHTS["rel"]}')
    set_seed(SEED)
    rel_model = JointCausalModel(**MODEL_CONFIG)
    opt = optim.AdamW(rel_model.parameters(), lr=TRAINING_CONFIG['learning_rate'], weight_decay=TRAINING_CONFIG['weight_decay'])
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.1, patience=2)

    rel_model, rel_hist = train_model(
        model=rel_model, train_dataloader=train_loader, val_dataloader=val_loader,
        optimizer=opt, num_epochs=TRAINING_CONFIG['num_epochs'], device=DEVICE,
        id2label_cls=id2label_cls, id2label_bio=id2label_bio, id2label_rel=id2label_rel,
        model_save_path=MODEL_PATHS['rel'], scheduler=sched,
        cls_class_weights=cls_w, bio_class_weights=bio_w, rel_class_weights=rel_w,
        patience_epochs=TRAINING_CONFIG['patience_epochs'], seed=SEED,
        max_grad_norm=TRAINING_CONFIG['gradient_clip_val'],
        eval_fn_metrics=make_eval_fn('rel'), print_report_fn=print_eval_report,
        task_loss_weights=TASK_WEIGHTS['rel'],
    )
    del rel_model
    torch.cuda.empty_cache()
else:
    print('REL model already trained — skipped')

print(f'REL-only best val F1: {max(rel_hist["val_overall_f1"]):.4f}')

REL model already trained — skipped
REL-only best val F1: 0.8687


---
## Training summary

In [9]:
histories = {'cls': cls_hist, 'bio': bio_hist, 'rel': rel_hist}

print(f'{"Model":<12} {"Best Val F1":>12}  {"Best Epoch":>10}  {"Early Stop":>10}')
print('-'*50)
for task, h in histories.items():
    best_f1 = max(h['val_overall_f1'])
    best_epoch = h['val_overall_f1'].index(best_f1) + 1
    stopped = len(h['val_overall_f1'])
    print(f'{task.upper():<12} {best_f1:>12.4f}  {best_epoch:>10}  {stopped:>10}')

Model         Best Val F1  Best Epoch  Early Stop
--------------------------------------------------
CLS                0.8117           9          19
BIO                0.4916          19          20
REL                0.8687           7          17


---
## Section 1: Per-task evaluation of separate models on test set

In [10]:
# Build test DataLoader
test_df = pd.read_csv(TEST_DATA_PATH)
test_dataset = CausalDataset(test_df, tokenizer_name='bert-base-uncased', max_length=DATASET_CONFIG['max_length'])
test_collator = CausalDatasetCollator(tokenizer=test_dataset.tokenizer)
test_loader = DataLoader(test_dataset, batch_size=TRAINING_CONFIG['batch_size'],
                         collate_fn=test_collator, shuffle=False, worker_init_fn=seed_worker)
print(f'Test batches: {len(test_loader)}')

Test batches: 29


In [11]:
def load_model(ckpt_path):
    m = JointCausalModel(**MODEL_CONFIG)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m.to(DEVICE).eval()
    return m

separate_results = {}
for task in ['cls', 'bio', 'rel']:
    print(f'\n=== {task.upper()}-only model ===')
    m = load_model(MODEL_PATHS[task])
    r = evaluate_model(m, test_loader, DEVICE, id2label_cls, id2label_bio, id2label_rel)
    separate_results[task] = r
    # Show only the active task metrics
    task_key = {'cls': 'task_cls', 'bio': 'task_bio', 'rel': 'task_relation'}[task]
    tm = r.get(task_key, {})
    if isinstance(tm, dict) and 'macro avg' in tm:
        ma = tm['macro avg']
        print(f'  Macro F1: {ma["f1-score"]:.4f}  P: {ma["precision"]:.4f}  R: {ma["recall"]:.4f}')
    del m
    torch.cuda.empty_cache()


=== CLS-only model ===


MRO for JointCausalModel: (<class 'jointlearning.model.JointCausalModel'>, <class 'torch.nn.modules.module.Module'>, <class 'huggingface_hub.hub_mixin.PyTorchModelHubMixin'>, <class 'huggingface_hub.hub_mixin.ModelHubMixin'>, <class 'object'>)


Evaluating:   0%|          | 0/29 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Evaluating:   3%|▎         | 1/29 [00:00<00:06,  4.37it/s]

...

Evaluating:  55%|█████▌    | 16/29 [00:00<00:00, 35.26it/s]

Evaluating:  69%|██████▉   | 20/29 [00:00<00:00, 36.49it/s]

Evaluating:  83%|████████▎ | 24/29 [00:00<00:00, 36.53it/s]

Evaluating:  97%|█████████▋| 28/29 [00:00<00:00, 35.51it/s]

  Macro F1: 0.8409  P: 0.8342  R: 0.8510


In [12]:
# Table 1
rows = []
task_names = {'cls': 'Classification', 'bio': 'BIO Tagging', 'rel': 'Relation Extraction'}
for task in ['cls', 'bio', 'rel']:
    if task not in separate_results:
        continue
    r = separate_results[task]
    tk = {'cls': 'task_cls', 'bio': 'task_bio', 'rel': 'task_relation'}[task]
    tm = r.get(tk, {})
    if isinstance(tm, dict) and 'macro avg' in tm:
        ma = tm['macro avg']
        rows.append({'Model': f'{task.upper()}-only', 'Task': task_names[task],
                     'Macro F1': ma['f1-score'], 'Precision': ma['precision'], 'Recall': ma['recall']})

t1 = pd.DataFrame(rows)
print('Table 1: Per-task evaluation of separate models (test set)')
print(t1.to_string(index=False))

Table 1: Per-task evaluation of separate models (test set)
   Model                Task  Macro F1  Precision   Recall
CLS-only      Classification  0.829505   0.832637 0.830689
BIO-only         BIO Tagging  0.489733   0.480664 0.532723
REL-only Relation Extraction  0.840938   0.834236 0.851002


---
## Section 2: Sequential pipeline evaluation

Chain CLS → BIO → REL with **no cleanup** between stages.

In [13]:
# Import the corrected sequential_predict and converter from evaluate_sequential
from evaluate_sequential import sequential_predict, sequential_to_doccano

print('Imported corrected sequential_predict + sequential_to_doccano from evaluate_sequential.py')

Imported corrected sequential_predict + sequential_to_doccano from evaluate_sequential.py


In [14]:
# Load models & tokenizer
cls_m = load_model(MODEL_PATHS['cls'])
bio_m = load_model(MODEL_PATHS['bio'])
rel_m = load_model(MODEL_PATHS['rel'])
tok = AutoTokenizer.from_pretrained(MODEL_CONFIG['encoder_name'])
texts = test_df['text'].tolist()

# Run sequential prediction
all_preds = []
BS = 32
for bi in tqdm(range((len(texts) + BS - 1)//BS), desc='Sequential predict'):
    batch = texts[bi*BS:(bi+1)*BS]
    all_preds.extend(sequential_predict(cls_m, bio_m, rel_m, batch, tok, DEVICE))

n_c = sum(1 for r in all_preds if r['causal'])
n_r = sum(len(r.get('relations',[])) for r in all_preds)
print(f'Predicted: {n_c}/{len(all_preds)} causal, {n_r} relations')

MRO for JointCausalModel: (<class 'jointlearning.model.JointCausalModel'>, <class 'torch.nn.modules.module.Module'>, <class 'huggingface_hub.hub_mixin.PyTorchModelHubMixin'>, <class 'huggingface_hub.hub_mixin.ModelHubMixin'>, <class 'object'>)


MRO for JointCausalModel: (<class 'jointlearning.model.JointCausalModel'>, <class 'torch.nn.modules.module.Module'>, <class 'huggingface_hub.hub_mixin.PyTorchModelHubMixin'>, <class 'huggingface_hub.hub_mixin.ModelHubMixin'>, <class 'object'>)


MRO for JointCausalModel: (<class 'jointlearning.model.JointCausalModel'>, <class 'torch.nn.modules.module.Module'>, <class 'huggingface_hub.hub_mixin.PyTorchModelHubMixin'>, <class 'huggingface_hub.hub_mixin.ModelHubMixin'>, <class 'object'>)


Sequential predict:   0%|          | 0/15 [00:00<?, ?it/s]

Sequential predict:   7%|▋         | 1/15 [00:00<00:02,  4.83it/s]

...

Sequential predict:  80%|████████  | 12/15 [00:02<00:00,  4.89it/s]

Sequential predict:  87%|████████▋ | 13/15 [00:02<00:00,  4.77it/s]

Sequential predict:  93%|█████████▎| 14/15 [00:02<00:00,  4.57it/s]

Sequential predict: 100%|██████████| 15/15 [00:02<00:00,  5.16it/s]

Predicted: 244/452 causal, 517 relations


In [15]:
# Convert to Doccano (using custom converter that preserves CLS decisions)
pred_df = sequential_to_doccano(all_preds)
csv_path = os.path.join(PREDICTIONS_DIR, 'sequential_predictions_doccano.csv')
pred_df.to_csv(csv_path, index=False)

gold_df = pd.read_csv(TEST_DATA_PATH)
print('Table 2: Sequential pipeline end-to-end results (corrected)\n' + '='*70)

table2_rows = []
for scenario in ['end_to_end', 'filtered_causal']:
    for eval_mode in ['discovery', 'coverage']:
        r = evaluate(gold_df, pred_df, scenario=scenario, eval_mode=eval_mode)
        display_results(r, title_prefix=f'Sequential | {scenario} | {eval_mode}')
        table2_rows.append({
            'Scenario': scenario, 'Eval Mode': eval_mode,
            'Task1 F1': r['Task1']['F1'], 'Task2 Macro': r['Task2_macro']['F1'],
            'Task3 F1': r['Task3']['F1'], 'Total Macro': r['Total_Macro']['F1'],
        })

t2 = pd.DataFrame(table2_rows)
print(t2.to_string(index=False))
t2.to_csv(os.path.join(PREDICTIONS_DIR, 'sequential_metrics.csv'), index=False)


SEQUENTIAL → DOCCANO CONVERSION COMPLETE
Total samples: 452
Causal: 244  Non-causal: 208
Total entities: 983
Total relations: 517
Table 2: Sequential pipeline end-to-end results (corrected)

            Sequential | end_to_end | discovery Results            

--- Task1 ---
  TP          :      192
  FP          :       45
  FN          :       29
  TN          :      186
  Precision   :   0.8101
  Recall      :   0.8688
  F1          :   0.8384
  Accuracy    :   0.8363
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.5781
    Recall      :   0.7088
    F1          :   0.6368
    TP          :      185
    FP          :      135
    FN          :       76
  Label: effect
    Precision   :   0.6188
    Recall      :   0.7778
    F1          :   0.6892
    TP          :      224
    FP          :      138
    FN          :       64

--- Task2_macro ---
  Precision   :   0.5985
  Recall      :   0.7433
  F1          :   0.6630
  TP          :      409
  FP     


           Sequential | filtered_causal | discovery Results           

--- Task1 ---
  TP          :      192
  FP          :       45
  FN          :       29
  TN          :      186
  Precision   :   0.8101
  Recall      :   0.8688
  F1          :   0.8384
  Accuracy    :   0.8363
  N           :      452

--- Task2 ---
  Label: cause
    Precision   :   0.7061
    Recall      :   0.8043
    F1          :   0.7520
    TP          :      185
    FP          :       77
    FN          :       45
  Label: effect
    Precision   :   0.7417
    Recall      :   0.8716
    F1          :   0.8014
    TP          :      224
    FP          :       78
    FN          :       33

--- Task2_macro ---
  Precision   :   0.7239
  Recall      :   0.8380
  F1          :   0.7767
  TP          :      409
  FP          :      155
  FN          :       78

--- Task3 ---
  TP          :       55
  FP          :      109
  FN          :      203
  Accuracy    :   0.1499
  Precision   :   0.3354
  Recal

---
## Section 3: Joint vs. Sequential comparison

Joint model numbers from `Notebooks/evaluation_report.md`
(bert-softmax, cls+span, threshold=0.5).

In [16]:
joint = pd.DataFrame([
    {'Model':'Joint','Scenario':'end_to_end','Eval Mode':'coverage',
     'Task1 F1':0.7981,'Task2 Macro':0.7136,'Task3 F1':0.5924,'Total Macro':0.7014},
    {'Model':'Joint','Scenario':'end_to_end','Eval Mode':'discovery',
     'Task1 F1':0.7981,'Task2 Macro':0.6904,'Task3 F1':0.5433,'Total Macro':0.6773},
    {'Model':'Joint','Scenario':'filtered_causal','Eval Mode':'coverage',
     'Task1 F1':0.7981,'Task2 Macro':0.8556,'Task3 F1':0.6941,'Total Macro':0.7826},
    {'Model':'Joint','Scenario':'filtered_causal','Eval Mode':'discovery',
     'Task1 F1':0.7981,'Task2 Macro':0.8361,'Task3 F1':0.6468,'Total Macro':0.7603},
])

seq = t2.copy()
seq.insert(0, 'Model', 'Sequential')

t3 = pd.concat([joint, seq], ignore_index=True)
print('Table 3: Joint vs Sequential')
print('='*70)
print(t3.to_string(index=False))

print('\nGap (= error propagation cost):')
for (_,j),(_,s) in zip(joint.iterrows(), seq.iterrows()):
    gap = j['Total Macro'] - s['Total Macro']
    print(f'  {j["Scenario"]:20s} {j["Eval Mode"]:10s}  gap = {gap:+.4f}  (Joint {j["Total Macro"]:.4f} - Seq {s["Total Macro"]:.4f})')

Table 3: Joint vs Sequential
     Model        Scenario Eval Mode  Task1 F1  Task2 Macro  Task3 F1  Total Macro
     Joint   end_to_end  coverage  0.798100     0.713600  0.592400     0.701400
     Joint   end_to_end discovery  0.798100     0.690400  0.543300     0.677300
     Joint filtered_causal  coverage  0.798100     0.855600  0.694100     0.782600
     Joint filtered_causal discovery  0.798100     0.836100  0.646800     0.760300
Sequential   end_to_end discovery  0.838428     0.663032  0.227273     0.576244
Sequential   end_to_end  coverage  0.838428     0.698850  0.256000     0.597759
Sequential filtered_causal discovery  0.838428     0.776732  0.260664     0.625274
Sequential filtered_causal  coverage  0.838428     0.809149  0.292237     0.646605

Gap (= error propagation cost):
  end_to_end        coverage    gap = +0.1252  (Joint 0.7014 - Seq 0.5762)
  end_to_end        discovery   gap = +0.0795  (Joint 0.6773 - Seq 0.5978)
  filtered_causal      coverage    gap = +0.1573  (Jo

In [17]:
# Cleanup
for m in [cls_m, bio_m, rel_m]:
    del m
torch.cuda.empty_cache()
print('Done. Models saved in models/  |  Predictions in predictions/')

Done. Models saved in models/  |  Predictions in predictions/
